# Data Science & Database Assignment: Session 8
## Topic: GROUP BY + HAVING

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `orders.db`  
**Core Concepts:** Aggregation (`COUNT`, `SUM`, `AVG`, `ROUND`), Grouping (`GROUP BY`), Group-Level Filtering (`HAVING`), Row Filtering vs Group Filtering (`WHERE` vs `HAVING`)  
**Status:** Completed & Pre-Executed  

---

### Logical Processing Order of Grouping & Aggregations

To truly understand how `GROUP BY` and `HAVING` interact with `WHERE`, consider the exact order in which SQL executes:

```text
1. FROM / JOIN     --> Identify source tables
2. WHERE           --> Filter individual rows (PRE-aggregation)
3. GROUP BY        --> Collapse remaining rows into distinct groups
4. AGGREGATE       --> Calculate COUNT(), SUM(), AVG(), MIN(), MAX() per group
5. HAVING          --> Filter summary groups based on aggregate conditions (POST-aggregation)
6. SELECT          --> Project output columns and aliases
7. ORDER BY        --> Sort final result rows
8. LIMIT           --> Restrict row count
```


## Task 1: Create `Orders` Table & Insert Sample E-Commerce Orders
> **Problem Statement:** Create a table called `Orders` with columns: `order_id`, `user_id`, `payment_method`, and `amount`. Insert at least 8 sample records representing different users and payment methods (like `UPI`, `Card`, `Wallet`, `COD`).

### Schema Definition
```sql
CREATE TABLE Orders (
    order_id INT PRIMARY KEY,
    user_id INT NOT NULL,
    payment_method VARCHAR(50) NOT NULL,
    amount DECIMAL(10, 2) NOT NULL
);
```


In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("orders.db")
cursor = conn.cursor()

# Drop existing table for idempotent execution
cursor.execute("DROP TABLE IF EXISTS Orders;")

# Create Orders table
cursor.execute("""
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    payment_method VARCHAR(50) NOT NULL,
    amount DECIMAL(10, 2) NOT NULL
);
""")

# Insert 9 realistic e-commerce / food orders
orders_data = [
    (1, 101, 'Card', 550.00),
    (2, 102, 'Card', 450.00),
    (3, 101, 'UPI', 320.00),
    (4, 103, 'UPI', 280.00),
    (5, 104, 'UPI', 420.00),
    (6, 102, 'COD', 180.00),
    (7, 103, 'COD', 220.00),
    (8, 104, 'Wallet', 150.00),
    (9, 101, 'Wallet', 250.00)
]

cursor.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?);", orders_data)
conn.commit()

df_orders = pd.read_sql_query("SELECT * FROM Orders;", conn)
print("Task 1 Output: Populated 'Orders' Table")
display(df_orders)


Task 1 Output: Populated 'Orders' Table


,order_id,user_id,payment_method,amount
0,1,101,Card,550
1,2,102,Card,450
2,3,101,UPI,320
3,4,103,UPI,280
4,5,104,UPI,420
5,6,102,COD,180
6,7,103,COD,220
7,8,104,Wallet,150
8,9,101,Wallet,250


## Task 2: Count Orders by Payment Method (Zomato Analytics Breakdown)
> **Problem Statement:** Write an SQL query to count how many orders were placed using each `payment_method` in the `Orders` table, similar to how Zomato shows payment breakdown in analytics.

### SQL Query:
```sql
SELECT 
    payment_method,
    COUNT(order_id) AS order_count
FROM Orders
GROUP BY payment_method;
```
* **Aggregation Function:** `COUNT(order_id)` computes the total number of transactions processed under each payment channel.
* **Grouping Criterion:** `GROUP BY payment_method` buckets the rows by distinct payment methods before aggregating.


In [2]:
query_t2 = """
SELECT 
    payment_method,
    COUNT(order_id) AS order_count
FROM Orders
GROUP BY payment_method
ORDER BY order_count DESC;
"""

df_t2 = pd.read_sql_query(query_t2, conn)
print("Task 2 Output: Order Volume Breakdown by Payment Method")
display(df_t2)


Task 2 Output: Order Volume Breakdown by Payment Method


,payment_method,order_count
0,UPI,3
1,Wallet,2
2,Card,2
3,COD,2


## Task 3: Total Spend by Each User (`SUM(amount) GROUP BY user_id`)
> **Problem Statement:** Write an SQL query to find the total amount spent by each `user_id` in the `Orders` table. Display `user_id` and their total spend.

### SQL Query:
```sql
SELECT 
    user_id,
    SUM(amount) AS total_spend
FROM Orders
GROUP BY user_id
ORDER BY total_spend DESC;
```
* **Customer Lifetime Value (CLV):** This fundamental query sums all purchases made by individual users to identify high-value VIP customers.


In [3]:
query_t3 = """
SELECT 
    user_id,
    SUM(amount) AS total_spend
FROM Orders
GROUP BY user_id
ORDER BY total_spend DESC;
"""

df_t3 = pd.read_sql_query(query_t3, conn)
print("Task 3 Output: Total Spend Grouped by User ID")
display(df_t3)


Task 3 Output: Total Spend Grouped by User ID


,user_id,total_spend
0,101,1120
1,102,630
2,104,570
3,103,500


## Task 4: Filter Groups with `HAVING AVG(amount) > 300`
> **Problem Statement:** Write an SQL query to show only those payment methods where the average order amount is greater than 300, using `GROUP BY` and `HAVING`.  
> *Hint: Use `AVG(amount)` in your `HAVING` clause.*

### SQL Query:
```sql
SELECT 
    payment_method,
    ROUND(AVG(amount), 2) AS avg_order_amount
FROM Orders
GROUP BY payment_method
HAVING AVG(amount) > 300;
```

### Why `WHERE` cannot be used here:
`WHERE AVG(amount) > 300` would throw an immediate SQL error (`misuse of aggregate function`). Why? Because `WHERE` evaluates row-by-row before any averages can be calculated! To filter based on the calculated summary value, `HAVING` is required.


In [4]:
query_t4 = """
SELECT 
    payment_method,
    ROUND(AVG(amount), 2) AS avg_order_amount
FROM Orders
GROUP BY payment_method
HAVING AVG(amount) > 300;
"""

df_t4 = pd.read_sql_query(query_t4, conn)
print("Task 4 Output: Payment Methods with Average Order Amount > 300")
display(df_t4)


Task 4 Output: Payment Methods with Average Order Amount > 300


,payment_method,avg_order_amount
0,Card,500.0
1,UPI,340.0


## Task 5: Deep Dive — `WHERE` vs `HAVING`
> **Problem Statement:** Explain the difference between `WHERE` and `HAVING` by giving one example query for each, using the `Orders` table. Your examples should show a scenario where `WHERE` and `HAVING` filter different things.

---

### Core Conceptual Differences

| Feature | `WHERE` Clause | `HAVING` Clause |
| :--- | :--- | :--- |
| **Filter Level** | **Row-level filter** (individual records) | **Group-level filter** (aggregated groups) |
| **Timing** | Evaluated **BEFORE** `GROUP BY` | Evaluated **AFTER** `GROUP BY` and aggregation |
| **Aggregate Functions** | **Cannot** use aggregates (`SUM`, `AVG`, `COUNT`) | **Can** use aggregates (`HAVING SUM(amount) > 500`) |
| **Performance** | Filters out rows early, reducing grouping work | Filters groups after calculation |

---

### Concrete Comparison Queries

#### 1. `WHERE` Example: Filtering Raw Transactions
> *"Count how many orders were over Rs. 250 for each payment method."*
```sql
SELECT 
    payment_method,
    COUNT(*) AS large_orders_count
FROM Orders
WHERE amount > 250          -- Filters individual rows BEFORE grouping
GROUP BY payment_method;
```

#### 2. `HAVING` Example: Filtering Aggregated Channels
> *"Show payment channels that brought in more than Rs. 600 in total revenue."*
```sql
SELECT 
    payment_method,
    SUM(amount) AS total_revenue
FROM Orders
GROUP BY payment_method
HAVING SUM(amount) > 600;   -- Filters summary groups AFTER calculating SUM
```

#### 3. Combined Pipeline (`WHERE` + `HAVING` in One Query)
> *"For orders of Rs. 200 or more, show payment methods whose combined revenue exceeds Rs. 500."*
```sql
SELECT 
    payment_method,
    COUNT(*) AS qualifying_orders,
    SUM(amount) AS total_revenue
FROM Orders
WHERE amount >= 200         -- Step 1: Discard small orders (< 200)
GROUP BY payment_method     -- Step 2: Group remaining orders
HAVING SUM(amount) > 500;   -- Step 3: Keep only high-volume groups
```


In [5]:
# 1. WHERE filter
df_where = pd.read_sql_query("""
SELECT 
    payment_method,
    COUNT(*) AS large_orders_count
FROM Orders
WHERE amount > 250
GROUP BY payment_method;
""", conn)

print("WHERE Example: Orders where amount > 250 per payment method")
display(df_where)

# 2. HAVING filter
df_having = pd.read_sql_query("""
SELECT 
    payment_method,
    SUM(amount) AS total_revenue
FROM Orders
GROUP BY payment_method
HAVING SUM(amount) > 600;
""", conn)

print("\nHAVING Example: Payment methods with total revenue > 600")
display(df_having)

# 3. Combined Pipeline
df_combined = pd.read_sql_query("""
SELECT 
    payment_method,
    COUNT(*) AS qualifying_orders,
    SUM(amount) AS total_revenue
FROM Orders
WHERE amount >= 200
GROUP BY payment_method
HAVING SUM(amount) > 500;
""", conn)

print("\nCombined Pipeline: (WHERE amount >= 200) + (HAVING SUM(amount) > 500)")
display(df_combined)

conn.close()


WHERE Example: Orders where amount > 250 per payment method


,payment_method,large_orders_count
0,Card,2
1,UPI,3



HAVING Example: Payment methods with total revenue > 600


,payment_method,total_revenue
0,Card,1000
1,UPI,1020



Combined Pipeline: (WHERE amount >= 200) + (HAVING SUM(amount) > 500)


,payment_method,qualifying_orders,total_revenue
0,Card,2,1000
1,UPI,3,1020


---

## 📌 Summary of Aggregation Rules Mastered

1. **Non-Aggregated Columns:** Every column in the `SELECT` list that is not wrapped in an aggregate function **must** appear in the `GROUP BY` clause.
2. **Filtering Summary Metrics:** Never attempt `WHERE SUM(x) > 10` or `WHERE COUNT(*) > 5`; always use `HAVING`.
3. **Execution Optimization:** When possible, filter rows with `WHERE` first to reduce the dataset volume before passing to `GROUP BY`.
